###Deep Normalization exercise
In this exercise you will learn to use normaliztion and use your research skills in order to fine tune a CNN.


Talk with the tutor about the following:
1.   What is the defenition of normalization?
2.   Why should you use normalization?

**IMPORTANT NOTE: BEFORE IMPLEMENTATION READ WHAT THE NORMALIZATION IS AND DECIDE WHETHER YOU SHOULD TAKE SOMETHING FROM GIT OR IMPLEMENT IT YOURSELF**


~Don Shaked

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torchvision
import numpy as np
import torch.optim as optim
import pickle
from torchvision import transforms
import random
from tqdm import tqdm

import os

import pandas as pd


zca_mat_path = "zca_mat.p"

results_path = "./results.csv"

print("CUDA is available: ", torch.cuda.is_available())

Mounted at /content/drive
CUDA is available:  True


In [ ]:
batch_size= # Write your own

epochs = 2 #2
num_experiments = 5 #5
# you can make the ratio 0.1 till you finished the debugging
ratio_of_data_to_keep = 1 #1

# columns' names
results_df = pd.DataFrame(columns=["Name", "loss_train_per_batch", "acc_train", "loss_test_per_batch", "acc_test", "run_time_sec"])

Write the forward of the following CNN:

In [ ]:
class CnnVanilla(nn.Module):
  def __init__(self):
    super().__init__() # (N, 1, 28, 28)
    self.conv1 = nn.Conv2d(1, 6, 5) # (N, 6, 24, 24)
    self.pool_1 = nn.MaxPool2d(2, 2) # (N, 6, 12, 12)
    self.conv2 = nn.Conv2d(6, 16, 5) # (N, 16, 8, 8)
    self.pool_2 = nn.MaxPool2d(2, 2) # (N, 16, 4, 4)
    self.fc1 = nn.Linear(256, 120) # (N, 120)
    self.fc2 = nn.Linear(120, 84) # (N, 84)
    self.fc3 = nn.Linear(84, 10) # (N, 10)

    self.sequential = nn.Sequential(
      self.conv1,
      self.pool_1,
      nn.ReLU(),
      self.conv2,
      self.pool_2,
      nn.ReLU(),
      nn.Flatten(1),
      self.fc1,
      nn.ReLU(),
      self.fc2,
      nn.ReLU(),
      self.fc3,
    )
  def forward(self, x):
    return self.sequential(x)



In [ ]:
def download_mnist_data(curr_transforms, train_flag, ratio_to_keep=ratio_of_data_to_keep):
    dataset_train = torchvision.datasets.MNIST('../data', train=train_flag, download=True, transform=curr_transforms)
    train_keep_size = int(ratio_to_keep * len(dataset_train))
    subset = torch.utils.data.random_split(dataset_train, [train_keep_size, len(dataset_train) - train_keep_size])[0]
    return subset

In [ ]:
def reproduce(seed=42):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

In [ ]:

def save_results(df, path):

  if os.path.isfile(path):

    saved_df = pd.read_csv(path, index_col=0)[results_df.columns]

    new_df = pd.concat([df, saved_df], axis=0)
    new_df.drop_duplicates(subset="Name", inplace=True)

  else:
    new_df = df

  new_df.to_csv(path)

  return new_df

The follwoing is a sketch of the training process, you'll need to change it accordingly each time for the different methods you'll apply. You'll need to provide batch size, number of workers and learning rate. **Supply learning rate appropriate to batch size**

In [ ]:
def train_model(model, trainloader, testloader, epochs=2, optimizer=None, criterion=None, verbose=False, device=None):
    #===Logistics====

    if optimizer is None:
      momentum = 0.9
      lr = 1e-2
      optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    if device is None:
      device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
      model = model.to(device)

    if criterion is None:
      criterion = nn.CrossEntropyLoss()

    #===Train Loop===
    for epoch in range(epochs):
      running_loss = 0.
      for batch, (input, labels) in enumerate(trainloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if verbose:
          if batch%(len(trainloader)//5) == (len(trainloader)//5)-1:
            print(f"Epoch:{epoch+1}, Batch:{batch+1}, Training Loss:{running_loss/(batch+1):.4f}")

    #===Test Loop===
    with torch.no_grad():
      train_loss = 0.
      train_correct_predictions = 0
      for batch, (input, labels) in enumerate(trainloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        train_correct_predictions += torch.sum(torch.argmax(output,dim=-1) == labels).item()
        train_loss += loss.item()

      train_acc = train_correct_predictions / len(trainloader.dataset)
      # Loss for all of a signle batch
      train_loss = train_loss / len(trainloader)

      test_loss = 0.
      test_correct_predictions = 0
      for batch, (input, labels) in enumerate(testloader):
        input, labels = input.to(device), labels.to(device)

        output = model(input)
        loss = criterion(output, labels)
        test_correct_predictions += torch.sum(torch.argmax(output,dim=-1) == labels).item()
        test_loss += loss.item()

      test_acc = test_correct_predictions / len(testloader.dataset)
      test_loss = test_loss / len(testloader)

    return train_loss, train_acc, test_loss, test_acc

In [ ]:
import time

def run_experiment(model_class, trainloader, testloader, epochs=2, num_experiments=10, momentum = 0.9, lr = 1e-2, optimizer_class=None, criterion=None, verbose=True, device=None, seed=42):
  #===Logistics====

  reproduce(seed)

  if device is None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  if criterion is None:
    criterion = nn.CrossEntropyLoss()

  if optimizer_class is None:
    optimizer_class = optim.SGD

  start_time = time.time()
  #===Loops===

  # For all experiments
  loss_train_arr = []
  acc_train_arr = []
  loss_test_arr = []
  acc_test_arr = []
  for experiment in range(num_experiments):
    model = model_class()
    model = model.to(device)
    optimizer = optimizer_class(model.parameters(), lr=lr, momentum=momentum)
    train_loss, train_acc, test_loss, test_acc = train_model(model, trainloader, testloader, epochs, optimizer, criterion, verbose=True, device=device)

    loss_train_arr.append(train_loss)
    acc_train_arr.append(train_acc)
    loss_test_arr.append(test_loss)
    acc_test_arr.append(test_acc)

    if verbose:
      print(f"Experiment:{experiment}, train_loss:{train_loss:.2f}, train_acc:{train_acc*100:.2f}%, test_loss:{test_loss:.2f}, test_acc:{test_acc*100:.2f}%")

  # Time for a single experiment
  time_sec = (time.time() - start_time) / num_experiments

  loss_train = np.mean(loss_train_arr)
  acc_train = np.mean(acc_train_arr)
  loss_test = np.mean(loss_test_arr)
  acc_test = np.mean(acc_test_arr)

  return loss_train, acc_train, loss_test, acc_test, time_sec

In [ ]:
"""
def train_model(dataset_train, dataset_test, batch_size, num_workers, lr, net_builder, seed=42):
  losses = []
  accuracies_train = []
  accuracies_test = []
  total_pred_train = 60000
  batches = total_pred_train / batch_size
  epochs = 2
  reproduce(seed)

  criterion = nn.CrossEntropyLoss()
  train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size,
                                              shuffle=True, num_workers=num_workers)
  test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size,
                                              shuffle=True, num_workers=num_workers)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  for _ in tqdm(range(10)):

        net = net_builder().to(device)
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
        for epoch in range(epochs):  # loop over the dataset multiple times
            for i, data in enumerate(train_loader, 0):
                # get the inputs; data is a list of [inputs, labels]
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                # zero the parameter gradients
                optimizer.zero_grad()
                # forward + backward + optimize
                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                # print statistics
                if epoch == epochs-1:
                  running_loss += loss.item() / batches
        losses.append(running_loss)
        correct_pred = 0

        with torch.no_grad():
            for data in train_loader:
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                _, predictions = torch.max(outputs, 1)
                # collect the correct predictions for each class
                correct_pred += torch.sum(predictions == labels).cpu().numpy()
            accuracies_train.append(correct_pred/total_pred_train)
            for data in test_loader:
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = net(inputs)
                _, predictions = torch.max(outputs, 1)
                # collect the correct predictions for each class
                correct_pred += torch.sum(predictions == labels).cpu().numpy()
            accuracies_test.append(correct_pred/total_pred)
"""


"\ndef train_model(dataset_train, dataset_test, batch_size, num_workers, lr, net_builder, seed=42):\n  losses = []\n  accuracies_train = []\n  accuracies_test = []\n  total_pred_train = 60000\n  batches = total_pred_train / batch_size\n  epochs = 2\n  reproduce(seed)\n\n  criterion = nn.CrossEntropyLoss()\n  train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size,\n                                              shuffle=True, num_workers=num_workers)\n  test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size,\n                                              shuffle=True, num_workers=num_workers)\n  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n  for _ in tqdm(range(10)):\n\n        net = net_builder().to(device)\n        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)\n        for epoch in range(epochs):  # loop over the dataset multiple times\n            for i, data in enumerate(train_loader, 0):\n       

####Normalize Activation -  population based methods
Here we will investigate a lot of dfferent population normalization methods.

The methods to implement are:



1.   no normalizaiton - baseline
2.   centering - 1d
3.   centering - all d
4.   scaling - 1d
5.   scaling - all d
6.   standardizing - 1d
7.   standardizing - all d
8.   whitenning - ZCA matrix is saved as pickle and in the folder for usage.

Write the complexity of each method.

Which normalization will work the best on the dataset, write it here (before running the experiment):

Talk with the tutor why you thought this way.




#Answer:#

**write answer**


Write each of the normaliztions as an augmentation, I provided the function of the whitening augmentation and provided the needed matrix in order for you to save time.

You'll need to calculate the statistics for each of the normalization methods.


In [ ]:
class Whitening:
    """Convert ndarrays in sample to Tensors."""

    def __init__(self, zca_mat_path):
      super().__init__()
      with open(zca_mat_path, 'rb') as file:
        self.zca_mat = pickle.load(file).type(torch.FloatTensor)


    def __call__(self, sample):
        f_sample = sample.flatten(1)
        proj_sample = torch.matmul(f_sample, self.zca_mat)

        return proj_sample.reshape((1, 28, 28))

class Normalize_1d:
    """Convert ndarrays in sample to Tensors."""

    def __init__(self, mean=None, std=None, eps=1e-2):

      pass


    def __call__(self, sample):
        pass


Show both train accuracy and validation accuracy.

As the lecture says you will train using 2 epochs.
Write the results of each model to a CSV file that you'll show your tutor.


#### Normalize Activation - As function methods
Here we will investigate a lot of dfferent population normalization methods. On the convolution layers.... ADD

The methods to implement are:



1.   no normalizaiton - baseline
2.   batch norm - regular
3.   batch norm - standardizing all d
4.   layer norm
5.   group norm - 2 groups
6.   batch whitening - Use the one from PyTorch


write the the pros and cons of each each method. Talk with them with your tutor, about when you'll use each one.

Here you'll need to change the architecture of the network. Write each one as a different class with an appropriate name.



In [ ]:
# create a dic of models

class CnnNormalization(nn.Module):
  def __init__(self, normalization_module=None, conv2d_module_class=None):
    # Both normalization_module and conv2d_module_class should be len 2 array-like objects
    super().__init__()
    pass


  def forward(self, x):
    pass

#### Normalize weights - As function methods
Here we will investigate 3 weight normalization methods.

The methods to implement are:



1.  Normalization Propagation
2.  Centered Weight Normalization
3.  Orthogonal Weight Normalization


What's the motivation behind each method ?
Here you'll need to change both architecture and the learning procedure



In [ ]:

class NormalizationPropagation2d(nn.Module):
  def __init__(self, in_channels):
    pass

  def forward(self, input):
    pass


class NormalizationPropagation2dGroupNorm(nn.Module):
  def __init__(self, group_size, in_channels):
    super().__init__()
    pass

  def forward(self, input):
    pass



#### Normalize Gradients - As function methods
Here we will investigate 2 gradients normalization methods.

The methods to implement are:



1.  Gradient centralization
2.  Layer-wise Adaptive Rate Scaling (LARS)


What's the motivation behind each method ?
Here you'll need to the learning procedure



#### Curse of Dimensionality

1. Here you'll combine the best methods from each section, and try to find other methods permutation that will be better. Combine the best results from each section.

2. After that try to find a better permutation.

3. In projects there are so many possible ways to solve the problem, so many permutation available.
What are the true possible permutation space of this problem?


#Answer for 3:#

**write answer**

In [ ]:
# Best combination results:


#### Final Words
You've learned a lot of different normalization methods in this exercise. If you want to review the material you've learned here you can watch the lecture by me "Normalization over 9000" it's in the Mador Lectures.

Hope you learned a lot,
Don Shaked